# 🌬️ NorthAir Intelligence — 01. Data Understanding
## VIZ-A-THON 2026: North India Air Quality Analytics Platform

### Overview & Problem Statement
North India experiences some of the most critical air pollution episodes globally, driven by seasonal meteorology, agricultural stubble burning, industrial emissions, and geographic valley effects across the Indo-Gangetic Plain.

In this notebook, we initialize the data foundation by loading and inspecting three core datasets provided for **VIZ-A-THON 2026**:
1. **`stations.csv`**: Metadata for 48 Continuous Ambient Air Quality Monitoring Stations (CAAQMS) across Chandigarh, Punjab, Haryana, and Delhi.
2. **`aqi_history_2017_2023.csv`**: Multi-year daily and hourly historical AQI values (102,744 records).
3. **`air_quality_hourly_2025.csv`**: Granular hourly measurements of 23 pollutant and meteorological parameters for the entire calendar year 2025 (414,255 records).

---


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 35)

# Add NorthAir source to path
sys.path.insert(0, str(Path("../src").resolve()))
from northair.data_loader import DataLoader

print("Environment initialized successfully.")


### 1. Data Loading via NorthAir DataLoader
We initialize `DataLoader`, which automatically detects raw CSV files in `data/raw/` and strips UTF-8 Byte Order Marks (`utf-8-sig`).


In [ ]:
loader = DataLoader(raw_dir=Path("../data/raw"))

# Check file presence
file_status = loader.check_files_exist()
print("Raw File Status:")
for k, v in file_status.items():
    print(f"  - {k}: {'EXISTS' if v else 'MISSING'}")

# Load datasets
stations_df = loader.load_stations()
aqi_hist_df = loader.load_aqi_history()
aq_2025_df = loader.load_air_quality_hourly_2025()

print(f"\nStations shape: {stations_df.shape}")
print(f"AQI History shape: {aqi_hist_df.shape}")
print(f"Air Quality 2025 shape: {aq_2025_df.shape}")


### 2. Inspecting Station Metadata (`stations.csv`)
Let us inspect the distribution of monitoring stations across the four North Indian administrative zones:
- **Chandigarh** (Union Territory)
- **Delhi** (National Capital Territory)
- **Haryana** (State)
- **Punjab** (State)


In [ ]:
summary_stations = loader.summarize_dataset(stations_df, name="stations.csv")
print(f"Total Stations: {summary_stations.rows}")
print(f"Missing Values: {summary_stations.total_missing_cells}")
print(f"Duplicate Rows: {summary_stations.duplicate_rows}\n")

display(stations_df.head(10))


In [ ]:
# Visualizing State-wise Station Distribution
plt.figure(figsize=(8, 4))
state_counts = stations_df['State'].value_counts()
ax = sns.barplot(x=state_counts.index, y=state_counts.values, palette='Blues_r')
plt.title("Monitoring Stations per State / UT (Total: 48)", fontsize=13, fontweight='bold')
plt.xlabel("State / UT")
plt.ylabel("Station Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11, xytext=(0, 3), textcoords='offset points')
plt.tight_layout()
plt.show()


### 3. Inspecting Historical AQI Data (`aqi_history_2017_2023.csv`)
Contains daily records with 24 individual hourly AQI columns (`00:00:00` to `23:00:00`).


In [ ]:
display(aqi_hist_df[['Station ID', 'State', 'City', 'Date', '00:00:00', '06:00:00', '12:00:00', '18:00:00']].head(5))
display(aqi_hist_df[['Station ID', 'State', 'City', 'Date', '00:00:00', '06:00:00', '12:00:00', '18:00:00']].tail(5))


### 4. Inspecting 2025 Hourly Air Quality Data (`air_quality_hourly_2025.csv`)
Contains 23 pollutant and environmental variables recorded at UTC timestamps.


In [ ]:
print("Columns in 2025 Hourly Dataset:")
for col in aq_2025_df.columns:
    print(f"  • {col}")

display(aq_2025_df[['Station ID', 'City', 'Timestamp', 'PM2.5 (µg/m³)', 'PM10 (µg/m³)', 'NO2 (µg/m³)', 'CO (mg/m³)', 'AT (°C)']].head(5))


### 5. Summary & Next Steps
- All 3 raw datasets were loaded successfully without inventing or modifying data.
- UTF-8 BOM encoding was seamlessly resolved.
- Next notebook: `02_data_quality_analysis.ipynb` will perform a deep audit of missing values, anomalies, and referential integrity.
